# F30K_T2S2_Seq Android Deployment Lock

Khóa gói encoder FP32 + decoder dynamic INT8 đã vượt Full Validation. Mô hình vẫn thuộc nhóm **thử nghiệm** do kết quả KD Flickr30K thấp hơn baseline huấn luyện.

## Input bắt buộc

1. output `MOB_F30K_T2S2_Seq_INT8_Full_Validation`;
2. output `MOB_F30K_T2S2_Seq_INT8_Preflight`.
3. output `MOB_F30K_T2S2_Seq_INT8_NonInferiority`.

Runtime: **CPU / Accelerator None / Internet Off**.


In [1]:
import csv, hashlib, json, os, shutil
from datetime import datetime, timezone
from pathlib import Path

MODEL_ID,TEACHER,STUDENT,METHOD,SEED,NEEDS_NONINF=('F30K_T2S2_Seq', 'BLIP Base', 'MobileViT-S', 'Sequence KD', 42, True)
VALIDATION_ID=f"MOB_{MODEL_ID}_INT8_Full_Validation"
PREFLIGHT_ID=f"MOB_{MODEL_ID}_INT8_Preflight"
EXPERIMENT_ID=f"MOB_{MODEL_ID}_Android_Lock"
INPUT_ROOT=Path(os.environ.get("KLCN_INPUT_ROOT","/kaggle/input"))
OUT=Path(os.environ.get("KLCN_OUTPUT_ROOT","/kaggle/working"))/EXPERIMENT_ID
PACKAGE=OUT/"android_package"; PACKAGE.mkdir(parents=True,exist_ok=True)
def sha256(path):
 h=hashlib.sha256()
 with Path(path).open("rb") as f:
  for b in iter(lambda:f.read(1024*1024),b""):h.update(b)
 return h.hexdigest()
def read(p):return json.loads(Path(p).read_text(encoding="utf-8"))
def write(p,o):Path(p).write_text(json.dumps(o,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
def one(name):
 found=sorted(INPUT_ROOT.rglob(name));assert found,(name,found)
 hashes={sha256(p) for p in found};assert len(hashes)==1,(name,found)
 return found[0]
val_path=one(VALIDATION_ID+"_RESULT.json"); val=read(val_path)
pre_path=one(PREFLIGHT_ID+"_RESULT.json"); pre=read(pre_path)
allowed_status={"passed"} if not NEEDS_NONINF else {"passed_with_caption_variation_review_required"}
assert val["model_id"]==MODEL_ID and val["status"] in allowed_status
assert val["test_set_used"] is False and pre["test_set_used"] is False
assert all(v["passed"] for v in val["gates"]["mixed_int8_quality"].values())
assert val["gates"]["fp32_reproduction"]["passed"] and val["gates"]["size"]["passed"]
assert pre["selected_candidate"]=="fp32_encoder_dynamic_int8_decoder"
noninf_path=None
if NEEDS_NONINF:
 noninf_id=f"MOB_{MODEL_ID}_INT8_NonInferiority"
 noninf_path=one(noninf_id+"_RESULT.json"); noninf=read(noninf_path)
 assert noninf["model_id"]==MODEL_ID and noninf["status"]=="passed" and noninf["passed"] is True
 assert noninf["confidence_interval_95"][0]>=noninf["noninferiority_margin"]
 assert noninf["test_set_used"] is False
copied=[]
for name,record in pre["artifacts"].items():
 src_candidates=[p for p in INPUT_ROOT.rglob(name) if p.is_file() and sha256(p)==record["sha256"]]
 assert src_candidates,(name,record["sha256"])
 src=sorted(src_candidates,key=lambda p:(len(p.parts),len(str(p))))[0]; dst=PACKAGE/name
 shutil.copy2(src,dst);assert dst.stat().st_size==record["bytes"] and sha256(dst)==record["sha256"]
 copied.append({"relative_path":str(dst.relative_to(OUT)).replace("\\","/"),"bytes":dst.stat().st_size,"sha256":sha256(dst)})
contract={"schema_version":"mobile_android_deployment_contract_v1","created_at_utc":datetime.now(timezone.utc).isoformat(),
 "status":"locked","experiment_id":EXPERIMENT_ID,"model_id":MODEL_ID,"teacher":TEACHER,"student":STUDENT,
 "dataset":val["dataset"],"method":METHOD,"seed":SEED,"model_group":"experimental",
 "precision":{"encoder":"FP32","decoder":"dynamic INT8"},"runtime_contract":pre["runtime_contract"],
 "decoding":val["decoding"],"metrics":val["metrics"],"selected_size":pre["selected_size"],
 "artifacts":copied,"source":{"validation_sha256":sha256(val_path),"preflight_sha256":sha256(pre_path),
 "noninferiority_sha256":sha256(noninf_path) if noninf_path else None},
 "validation_gate_passed":True,"test_set_used":False,"android_assets_authorized":True,
 "next_phase_authorized":f"MOB_{MODEL_ID}_Android_Integration"}
contract_path=PACKAGE/"deployment_contract.json";write(contract_path,contract)
files=copied+[{"relative_path":str(contract_path.relative_to(OUT)).replace("\\","/"),"bytes":contract_path.stat().st_size,"sha256":sha256(contract_path)}]
write(OUT/"MANIFEST.json",{"schema_version":"mobile_android_lock_manifest_v1","status":"complete","artifacts":files,"test_set_used":False})
lock_path=OUT/f"{EXPERIMENT_ID}_COMPLETE.json";write(lock_path,{"status":"complete","experiment_id":EXPERIMENT_ID,
 "model_id":MODEL_ID,"android_assets_authorized":True,"deployment_contract_sha256":sha256(contract_path),"test_set_used":False,
 "next_phase_authorized":contract["next_phase_authorized"]})
print("Status                : LOCKED")
print("Model                 :",MODEL_ID)
print("Package MiB           :",pre["selected_size"]["selected_mib"])
print("Android integration   : AUTHORIZED")
print("Next                  :",contract["next_phase_authorized"])


Status                : LOCKED
Model                 : F30K_T2S2_Seq
Package MiB           : 26.51373863220215
Android integration   : AUTHORIZED
Next                  : MOB_F30K_T2S2_Seq_Android_Integration
